In [1]:
# Final Biomarker Summary

#Objective:

#Identify biomarkers that are:

#- statistically different between groups;
#- predictive of phenoconversion;
#- robust across machine learning models.

In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")

RESULTS_DIR = PROJECT_ROOT / "results"

In [3]:
# on charge els résultats de l'ANCOVA
ancova = pd.read_csv(
    RESULTS_DIR /
    "ancova" /
    "repeated_measures_ancova_group_effects_fdr.csv"
)

ancova.head()

,effect,sum_sq,df,F,PR(>F),variable,p_fdr,significant_fdr
0,C(conversion_status),1.092981,1.0,0.510955,0.475367,density_sw,0.500387,False
1,C(conversion_status),30688.310905,1.0,0.981298,0.322800,total_N2_sw_count,0.358666,False
2,C(conversion_status),400792.622861,1.0,8.714675,0.003445,total_N3_sw_count,0.005742,True
3,C(conversion_status),653749.668285,1.0,5.905269,0.015773,total_sw_count,0.022533,True
4,C(conversion_status),0.001672,1.0,0.170645,0.679885,total_N2_sw_sec,0.679885,False


In [4]:
# on extrait les variables significatives 
sig_ancova = ancova[
    ancova["PR(>F)"] < 0.05
]

sig_ancova

,effect,sum_sq,df,F,PR(>F),variable,p_fdr,significant_fdr
2,C(conversion_status),400792.622861,1.0,8.714675,3.445014e-03,total_N3_sw_count,5.741690e-03,True
3,C(conversion_status),653749.668285,1.0,5.905269,1.577297e-02,total_sw_count,2.253282e-02,True
7,C(conversion_status),0.566721,1.0,61.540012,1.178927e-13,total_trans_freq_Hz,2.357855e-12,True
8,C(conversion_status),22621.947011,1.0,31.798958,4.512262e-08,total_slope_min_max,1.289218e-07,True
9,C(conversion_status),15346.917563,1.0,17.433322,4.082499e-05,total_slope_0_min,1.020625e-04,True
10,C(conversion_status),0.289700,1.0,55.283906,1.569863e-12,total_freq_Hz,1.569863e-11,True
11,C(conversion_status),370.342963,1.0,7.790400,5.648109e-03,total_pkpk_amp_uV,8.689399e-03,True
12,C(conversion_status),0.209873,1.0,32.795083,2.862416e-08,total_N2_freq_Hz,9.541386e-08,True
13,C(conversion_status),0.191580,1.0,39.560547,1.964092e-09,total_N3_freq_Hz,9.820461e-09,True
14,C(conversion_status),0.393239,1.0,33.976490,1.672601e-08,total_N2_trans_freq_Hz,6.690404e-08,True


In [5]:
# on charge les biomarqueurs les plus robustes
common = pd.read_csv(
    RESULTS_DIR /
    "ml" /
    "common_biomarkers.csv"
)

common

,feature
0,central_total_N3_pkpk_amp_uV
1,central_total_freq_Hz
2,central_total_trans_freq_Hz
3,frontal_total_slope_0_min
4,frontal_total_sw_count
5,occipital_total_N2_pkpk_amp_uV
6,occipital_total_N3_pkpk_amp_uV
7,occipital_total_slope_0_min
8,parietal_total_N2_freq_Hz
9,parietal_total_sw_count


In [6]:
#on a comparer les suffixes dans les deux analyses ANCOVA et biomarqueurs du classifier 
ancova_features = set(
    sig_ancova["variable"]
)

ancova_features

{'total_N2_freq_Hz',
 'total_N2_pkpk_amp_uV',
 'total_N2_slope_0_min',
 'total_N2_trans_freq_Hz',
 'total_N3_freq_Hz',
 'total_N3_pkpk_amp_uV',
 'total_N3_sw_count',
 'total_N3_trans_freq_Hz',
 'total_freq_Hz',
 'total_pkpk_amp_uV',
 'total_slope_0_min',
 'total_slope_min_max',
 'total_sw_count',
 'total_trans_freq_Hz'}

In [7]:
# on identifie les biomarqueurs robustes qui sont aussi des varibales 
#singificatives dan sl'ANCOVA 
matches = []

for feature in common["feature"]:

    for ancova_var in ancova_features:

        if ancova_var in feature:

            matches.append(
                {
                    "feature": feature,
                    "ancova_variable": ancova_var
                }
            )

matches = pd.DataFrame(matches)

matches

,feature,ancova_variable
0,central_total_N3_pkpk_amp_uV,total_N3_pkpk_amp_uV
1,central_total_freq_Hz,total_freq_Hz
2,central_total_trans_freq_Hz,total_trans_freq_Hz
3,frontal_total_slope_0_min,total_slope_0_min
4,frontal_total_sw_count,total_sw_count
5,occipital_total_N2_pkpk_amp_uV,total_N2_pkpk_amp_uV
6,occipital_total_N3_pkpk_amp_uV,total_N3_pkpk_amp_uV
7,occipital_total_slope_0_min,total_slope_0_min
8,parietal_total_N2_freq_Hz,total_N2_freq_Hz
9,parietal_total_sw_count,total_sw_count


In [8]:
# on enregistre les résulatst pour produire final_biomarker_summary.csv
matches.to_csv(
    RESULTS_DIR /
    "ml" /
    "final_biomarker_summary.csv",
    index=False
)